# UGA Grad Survivor v2 — Balance Simulation & Real-World Calibration

This notebook runs Monte Carlo simulations of the game engine and compares the resulting ending distributions against real-world PhD attrition data.

**Purpose:** Verify that the game's difficulty and ending distribution roughly matches the *shape* of real PhD outcomes.

**Important caveat:** The "reality" column is synthesized from multiple sources — no single dataset gives a clean breakdown of PhD non-completion by category. See the Data Sources section below for exactly what was used and what was estimated.

## Data Sources & Methodology

### What we know with confidence (cited data):

| Claim | Value | Source | Confidence |
|-------|-------|--------|-------------|
| PhD completion rate (life sciences, 10yr) | ~63% | [CGS PhD Completion Project](https://cgsnet.org/data-insights/diversity-equity-inclusiveness/degree-completion) — 49,000+ students, 30 universities | **High** |
| PhD completion rate (STEM overall) | ~55-64% | [CGS baseline data](https://cgsnet.org/wp-content/uploads/2022/01/phd_completion_and_attrition_analysis_of_baseline_demographic_data-2.pdf) | **High** |
| Grad students reporting moderate-severe stress | >75% | [Nature 2022 global survey](https://www.nature.com/articles/d41586-022-03394-0) | **High** |
| Grad students with depression/anxiety symptoms | ~40-50% | [PMC systematic review](https://pmc.ncbi.nlm.nih.gov/articles/PMC7483179/) | **High** |
| Grad students reporting financial stress | ~62% | [Nature 2022 survey](https://www.nature.com/articles/d41586-022-03394-0) | **High** |
| STEM PhDs in private sector (post-graduation) | ~42-47% | [PNAS 2019](https://www.pnas.org/doi/10.1073/pnas.1820079116) | **High** |
| Satisfaction declined since starting PhD | ~50% | [Nature 2022 survey](https://www.nature.com/articles/d41586-022-03394-0) | **High** |

### What we estimated (synthesized from multiple sources):

| Estimate | Value | Basis | Confidence |
|----------|-------|-------|-------------|
| Mastered out (left with master's) | ~15-20% of starters | CGS data shows most attrition is in years 2-4; "mastered out" is the most common exit category in program-level data | **Medium** — range is reasonable but exact % varies by field |
| Left for industry (voluntary, pre-completion) | ~8-12% of starters | Inferred from STEM career transition studies + the fact that ~45% of completers go to industry anyway | **Low-Medium** — hard to separate "left for industry" from "mastered out for industry" |
| Burned out (health/mental health withdrawal) | ~5-8% of starters | Nature survey + [PMC burnout studies](https://pmc.ncbi.nlm.nih.gov/articles/PMC10163855/) — high stress rates but actual withdrawal is a subset | **Medium** — burnout is common, formal withdrawal for it is less common |
| ABD (all but dissertation) | ~3-5% of starters | Estimated from attrition timing data — a small group makes it to candidacy but never defends | **Low** — ABD is poorly tracked in most datasets |
| Pure financial failure | ~2-4% of starters | Financial stress is common (~62%) but is almost always an amplifier of other factors, not the sole cause | **Medium** — supported by literature but exact rate is uncertain |

### Key insight for game design:
The literature consistently shows that financial stress is the **background condition** that amplifies burnout and attrition, not usually the direct cause of departure. This informed the "financial stress mechanic" in the game where low Wallet drains Mind and Body rather than causing instant death.

### What the game targets vs reality:
The game is **intentionally harder** than real life (~10% random defend rate vs ~63% real completion) because:
1. The game needs to be challenging enough to show players multiple endings
2. "Strategic play" (paying attention to stat bars) should feel rewarding (hitting ~70-98%)
3. Death = progress in the Reigns design philosophy — seeing different endings IS the game

## Setup: Extract Game Data via Node.js

In [ ]:
import subprocess
import json
import random
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from collections import Counter, defaultdict
from copy import deepcopy

# Node.js extraction script
extract_script = """
const fs = require('fs');
const html = fs.readFileSync('../index.html', 'utf8');
const scriptMatch = html.match(/<script>([\\s\\S]*)<\\/script>/);
if (!scriptMatch) { console.error('No script found'); process.exit(1); }
let script = scriptMatch[1];

// Convert const/let to var for eval compatibility
script = script.replace(/\\bconst\\b/g, 'var').replace(/\\blet\\b/g, 'var');

// Remove initialization calls
script = script.replace(/loadSave\\(\\);/g, '');
script = script.replace(/^render\\(\\);$/m, '');

// Mock DOM/storage
var mockEl = { 
  innerHTML:'', 
  style:{}, 
  classList:{add:function(){},remove:function(){}},
  addEventListener:function(){},
  offsetWidth: 400,
  offsetHeight: 600
};
global.document = { 
  getElementById:function(){ return mockEl; },
  addEventListener: function(){},
  body: { classList: {add:function(){}} }
};
global.localStorage = { 
  getItem:function(){ return null; }, 
  setItem:function(){},
  clear: function(){}
};
global.window = { addEventListener: function(){}, requestAnimationFrame: function(cb){cb();} };
global.navigator = { clipboard:{writeText:function(){ return Promise.resolve(); }} };
global.TouchEvent = function(){};
global.PointerEvent = function(){};

// Eval the game script
eval(script);

// Serialize cards (extract only essential fields for simulation)
function serializeCards(cards) {
  if (!cards) return [];
  return cards.map(function(c) {
    var obj = {
      id: c.id || '',
      tag: c.tag || '',
      eL: c.eL || null,
      eR: c.eR || null
    };
    if (c.milestone) obj.milestone = true;
    if (c.semester) obj.semester = c.semester;
    if (c.sets) obj.sets = c.sets;
    if (c.requires) obj.requires = c.requires;
    if (c.exclusive) obj.exclusive = c.exclusive;
    if (c.industry) obj.industry = c.industry;
    return obj;
  });
}

var data = {
  PHASE1: serializeCards(typeof PHASE1_CARDS !== 'undefined' ? PHASE1_CARDS : []),
  PHASE2: serializeCards(typeof PHASE2_CARDS !== 'undefined' ? PHASE2_CARDS : []),
  PHASE3: serializeCards(typeof PHASE3_CARDS !== 'undefined' ? PHASE3_CARDS : []),
  UNIVERSAL: serializeCards(typeof UNIVERSAL_CARDS !== 'undefined' ? UNIVERSAL_CARDS : []),
  CALLBACKS: serializeCards(typeof CALLBACK_CARDS !== 'undefined' ? CALLBACK_CARDS : []),
  ORGANIZER: serializeCards(typeof ORGANIZER_CARDS !== 'undefined' ? ORGANIZER_CARDS : []),
  MILESTONES: serializeCards(typeof MILESTONE_CARDS !== 'undefined' ? MILESTONE_CARDS : []),
  ARCHETYPES: {}
};

if (typeof ARCHETYPE_DATA !== 'undefined') {
  Object.keys(ARCHETYPE_DATA).forEach(function(k) {
    data.ARCHETYPES[k] = ARCHETYPE_DATA[k].st || ARCHETYPE_DATA[k];
  });
}

process.stdout.write(JSON.stringify(data));
"""

# Run Node extraction
result = subprocess.run(['node', '-e', extract_script], capture_output=True, text=True, cwd='.')
if result.returncode != 0:
    print("STDERR:", result.stderr)
    raise Exception(f"Node extraction failed: {result.stderr}")

game_data = json.loads(result.stdout)

# Load card pools
PHASE1_CARDS = game_data['PHASE1']
PHASE2_CARDS = game_data['PHASE2']
PHASE3_CARDS = game_data['PHASE3']
UNIVERSAL_CARDS = game_data['UNIVERSAL']
CALLBACK_CARDS = game_data['CALLBACKS']
ORGANIZER_CARDS = game_data['ORGANIZER']
MILESTONE_CARDS = game_data['MILESTONES']
ARCHETYPE_DATA = game_data['ARCHETYPES']

print(f"Loaded card pools:")
print(f"  PHASE1:     {len(PHASE1_CARDS)} cards")
print(f"  PHASE2:     {len(PHASE2_CARDS)} cards")
print(f"  PHASE3:     {len(PHASE3_CARDS)} cards")
print(f"  UNIVERSAL:  {len(UNIVERSAL_CARDS)} cards")
print(f"  CALLBACKS:  {len(CALLBACK_CARDS)} cards")
print(f"  ORGANIZER:  {len(ORGANIZER_CARDS)} cards")
print(f"  MILESTONES: {len(MILESTONE_CARDS)} cards")
print(f"  Total:      {sum(len(l) for l in [PHASE1_CARDS, PHASE2_CARDS, PHASE3_CARDS, UNIVERSAL_CARDS, CALLBACK_CARDS, ORGANIZER_CARDS, MILESTONE_CARDS])}")
print(f"\nArchetypes: {list(ARCHETYPE_DATA.keys())}")
for name, stats in ARCHETYPE_DATA.items():
    print(f"  {name:12s}: M={stats['mind']:2d} B={stats['body']:2d} W={stats['wallet']:2d} Bo={stats['bonds']:2d}")

Loaded card pools:
  PHASE1:     19 cards
  PHASE2:     56 cards
  PHASE3:     27 cards
  UNIVERSAL:  11 cards
  CALLBACKS:  6 cards
  ORGANIZER:  3 cards
  MILESTONES: 6 cards
  Total:      128

Archetypes: ['overachiever', 'imposter', 'coaster', 'believer', 'organizer']
  overachiever: M=50 B=45 W=65 Bo=55
  imposter    : M=35 B=55 W=65 Bo=60
  coaster     : M=65 B=65 W=55 Bo=50
  believer    : M=55 B=50 W=55 Bo=50
  organizer   : M=45 B=50 W=55 Bo=65


## Game Engine (Python port)

In [ ]:
def get_phase(semester):
    """Return phase number (1, 2, or 3) based on semester."""
    if semester <= 2:
        return 1
    elif semester <= 6:
        return 2
    else:
        return 3

def get_milestone(semester):
    """Get milestone card for this semester if it exists."""
    for m in MILESTONE_CARDS:
        if m.get('semester') == semester:
            return m
    return None

def draw_card(state):
    """Draw a card from the appropriate pool."""
    # Check for pending milestone (every 5 cards)
    if state['card_count'] >= 5 and state['next_milestone']:
        ms = state['next_milestone']
        state['next_milestone'] = None
        state['card_count'] = 0
        return ms
    
    # Reset card count if we've drawn 5 without hitting a milestone
    if state['card_count'] >= 5:
        state['card_count'] = 0
    
    # Get phase and build pool
    phase = get_phase(state['semester'])
    if phase == 1:
        pool = list(PHASE1_CARDS)
    elif phase == 2:
        pool = list(PHASE2_CARDS)
    else:
        pool = list(PHASE3_CARDS)
    
    # Add universal cards
    pool.extend(UNIVERSAL_CARDS)
    
    # Filter already-seen cards
    pool = [c for c in pool if c['id'] not in state['memory']]
    
    # Add eligible callbacks (cards that fire when prerequisites are seen)
    for cb in CALLBACK_CARDS:
        if cb['id'] in state['memory']:
            continue
        reqs = cb.get('requires', [])
        if reqs and all(r in state['memory'] for r in reqs):
            pool.append(cb)
    
    # Add organizer cards (organizer archetype only)
    if state['archetype'] == 'organizer':
        for oc in ORGANIZER_CARDS:
            if oc['id'] not in state['memory']:
                pool.append(oc)
    
    if not pool:
        return None
    
    return random.choice(pool)

def apply_perk(archetype, stat, delta, card_tag=''):
    """Apply archetype-specific perk modifiers to stat deltas."""
    if archetype == 'imposter':
        # Imposter: bonds losses are halved (better at relationships)
        if stat == 'bonds' and delta < 0:
            return delta // 2
        # Imposter: double penalty on existential events
        if stat == 'mind' and delta < 0 and 'Existential' in card_tag:
            return delta * 2
    elif archetype == 'believer':
        # Believer: wallet losses are 30% softer (optimism/faith cushion)
        if stat == 'wallet' and delta < 0:
            return int(delta * 0.7)  # 30% reduction in loss
    elif archetype == 'organizer':
        # Organizer: reduce losses by 3 when bonds are high
        if delta < 0:
            return max(delta + 3, -3)  # Can't reduce loss by more than 3
    return delta

def simulate_game(archetype, strategy='random', seed=None):
    """
    Simulate one complete game run.
    
    Strategies:
    - 'random':     50/50 coin flip
    - 'left_bias':  70% left, 30% right
    - 'right_bias': 30% left, 70% right
    - 'safe':       pick choice with higher net stat sum
    - 'yolo':       pick choice with largest absolute swing
    """
    if seed is not None:
        random.seed(seed)
    
    # Initialize state from archetype template
    st = dict(ARCHETYPE_DATA[archetype])
    state = {
        'archetype': archetype,
        'st': st,
        'semester': 1,
        'card_count': 0,
        'total_cards': 0,
        'memory': [],
        'next_milestone': get_milestone(1),
        'ending': None,
        'cause': None,
    }
    
    for turn in range(200):  # Safety limit to prevent infinite loops
        card = draw_card(state)
        if card is None:
            state['ending'] = 'stuck'
            break
        
        # Determine choice direction based on strategy
        if strategy == 'random':
            side = 'left' if random.random() > 0.5 else 'right'
        elif strategy == 'left_bias':
            side = 'left' if random.random() > 0.3 else 'right'
        elif strategy == 'right_bias':
            side = 'left' if random.random() > 0.7 else 'right'
        elif strategy == 'safe':
            l_sum = sum((card.get('eL') or {}).values()) if card.get('eL') else 0
            r_sum = sum((card.get('eR') or {}).values()) if card.get('eR') else 0
            side = 'left' if l_sum >= r_sum else 'right'
        elif strategy == 'yolo':
            l_vals = list((card.get('eL') or {}).values()) if card.get('eL') else [0]
            r_vals = list((card.get('eR') or {}).values()) if card.get('eR') else [0]
            l_max = max([abs(v) for v in l_vals]) if l_vals else 0
            r_max = max([abs(v) for v in r_vals]) if r_vals else 0
            side = 'left' if l_max >= r_max else 'right'
        else:
            side = 'left'
        
        # Handle industry ending (leaving for industry/career pivot)
        if side == 'left' and card.get('industry'):
            state['ending'] = card['industry']
            state['cause'] = 'Career Choice'
            break
        
        # Apply effects
        effects = card.get('eL' if side == 'left' else 'eR')
        if effects:
            tag = card.get('tag', '')
            for stat, delta in effects.items():
                # Apply archetype perks
                modified_delta = apply_perk(state['archetype'], stat, delta, tag)
                
                # Clamp to [0, 100]
                st[stat] = max(0, min(100, st[stat] + modified_delta))
            
            # Post-effect archetype bonuses
            if state['archetype'] == 'overachiever':
                # Overachiever: if any stat is very low, slightly boost a mid-range stat
                weak = [s for s in ['mind', 'body', 'wallet', 'bonds'] if st[s] < 20]
                if weak:
                    candidates = [s for s in ['mind', 'body', 'wallet', 'bonds'] if st[s] >= 20]
                    if not candidates:
                        candidates = ['mind', 'body', 'wallet', 'bonds']
                    pick = random.choice(candidates)
                    st[pick] = min(100, st[pick] + 5)
            
            if state['archetype'] == 'coaster':
                # Coaster: mind and body have a floor of 15 (lazy but stable)
                st['mind'] = max(15, st['mind'])
                st['body'] = max(15, st['body'])
            
            # Financial stress mechanic: low wallet drains mind and body each turn
            if st['wallet'] < 5:
                st['wallet'] = 5  # Absolute floor (can't go to 0)
            if st['wallet'] < 20:
                st['mind'] = max(0, st['mind'] - 3)
                st['body'] = max(0, st['body'] - 2)
            
            # Believer: periodic morale boosts
            if state['archetype'] == 'believer' and state['total_cards'] > 0 and state['total_cards'] % 5 == 0:
                st['mind'] = min(100, st['mind'] + 10)
        
        # Record memory (flags from card sets)
        if card.get('sets'):
            for flag in card['sets']:
                if flag not in state['memory']:
                    state['memory'].append(flag)
        state['memory'].append(card['id'])
        
        state['total_cards'] += 1
        state['card_count'] += 1
        
        # Check ending conditions
        phase = get_phase(state['semester'])
        
        if st['wallet'] <= 0:
            state['ending'] = 'broke'
            state['cause'] = 'Wallet'
            break
        if st['body'] <= 0:
            state['ending'] = 'burned_out'
            state['cause'] = 'Body'
            break
        if st['mind'] <= 0:
            # Mind loss in early phases = mastered out; later = ABD
            state['ending'] = 'mastered_out' if phase <= 2 else 'abd'
            state['cause'] = 'Mind'
            break
        if st['bonds'] <= 0:
            # Bonds loss in early phases = mastered out; later = ABD
            state['ending'] = 'mastered_out' if phase <= 2 else 'abd'
            state['cause'] = 'Bonds'
            break
        
        # Victory: successfully defended
        if 'defended' in state['memory']:
            state['ending'] = 'defended'
            break
        
        # Time limit: past semester 10 without defending = ABD
        if state['semester'] > 10 and 'defended' not in state['memory']:
            state['ending'] = 'abd'
            state['cause'] = 'Time'
            break
        
        # Advance semester after 5 cards
        if state['card_count'] >= 5:
            state['semester'] += 1
            state['next_milestone'] = get_milestone(state['semester'])
    
    # Fallback ending
    if state['ending'] is None:
        state['ending'] = 'stuck'
    
    return {
        'ending': state['ending'],
        'cause': state['cause'],
        'semester': state['semester'],
        'total_cards': state['total_cards'],
        'final_stats': dict(st),
        'archetype': archetype,
        'strategy': strategy,
    }

# Quick test
result = simulate_game('overachiever', 'random', seed=42)
print(f"Test run: {result['ending']} at semester {result['semester']} ({result['total_cards']} cards)")
print(f"Final stats: M={result['final_stats']['mind']} B={result['final_stats']['body']} "
      f"W={result['final_stats']['wallet']} Bo={result['final_stats']['bonds']}")

Test run: mastered_out at semester 5 (21 cards)
Final stats: M=0 B=20 W=82 Bo=90


## Run Simulation (5,000 runs)

In [ ]:
RUNS_PER_COMBO = 200
ARCHETYPES_LIST = list(ARCHETYPE_DATA.keys())
STRATEGIES = ['random', 'left_bias', 'right_bias', 'safe', 'yolo']

print(f"Running {RUNS_PER_COMBO} runs × {len(ARCHETYPES_LIST)} archetypes × {len(STRATEGIES)} strategies = {RUNS_PER_COMBO * len(ARCHETYPES_LIST) * len(STRATEGIES)} total simulations...")

all_results = []
for arch in ARCHETYPES_LIST:
    for strat in STRATEGIES:
        for i in range(RUNS_PER_COMBO):
            result = simulate_game(arch, strat)
            all_results.append(result)

print(f"\nCompleted {len(all_results)} simulations")
print(f"Unique endings found: {sorted(set(r['ending'] for r in all_results))}")

Running 200 runs × 5 archetypes × 5 strategies = 5000 total simulations...

Completed 5000 simulations
Unique endings found: ['abd', 'burned_out', 'defended', 'left_for_industry', 'mastered_out']


## Results: Game vs Reality

In [ ]:
# Overall ending counts
ending_counts = Counter(r['ending'] for r in all_results)
total = len(all_results)

# Reality targets (with confidence notes)
reality = {
    'defended':          {'pct': 63, 'conf': 'High',   'source': 'CGS PhD Completion Project'},
    'mastered_out':      {'pct': 17, 'conf': 'Medium', 'source': 'CGS + field estimates'},
    'left_for_industry': {'pct': 10, 'conf': 'Low-Med','source': 'STEM career transition studies'},
    'burned_out':        {'pct':  6, 'conf': 'Medium', 'source': 'Nature 2022 survey + PMC'},
    'abd':               {'pct':  4, 'conf': 'Low',    'source': 'Attrition timing estimates'},
    'broke':             {'pct':  3, 'conf': 'Medium', 'source': 'Financial stress literature'},
}

print(f"{'Ending':<22} {'Game':>6} {'Reality':>8} {'Conf':>10} {'Source'}")
print('-' * 85)
for ending in ['defended', 'mastered_out', 'left_for_industry', 'abd', 'burned_out', 'broke']:
    game_pct = ending_counts.get(ending, 0) / total * 100
    real = reality.get(ending, {})
    print(f"{ending:<22} {game_pct:5.1f}% {real.get('pct','?'):>6}% {real.get('conf','?'):>10}   {real.get('source','')}")

# Bar chart
fig, ax = plt.subplots(1, 1, figsize=(10, 5))
endings_order = ['defended', 'mastered_out', 'left_for_industry', 'abd', 'burned_out', 'broke']
labels = ['Defended', 'Mastered\nOut', 'Left for\nIndustry', 'ABD', 'Burned\nOut', 'Broke']
game_pcts = [ending_counts.get(e, 0) / total * 100 for e in endings_order]
real_pcts = [reality[e]['pct'] for e in endings_order]

x = np.arange(len(endings_order))
width = 0.35

bars1 = ax.bar(x - width/2, game_pcts, width, label='Game (5,000 runs)', color='#d4a017', alpha=0.9)
bars2 = ax.bar(x + width/2, real_pcts, width, label='Reality (estimated)', color='#2d6a4f', alpha=0.9)

ax.set_ylabel('Percentage')
ax.set_title('UGA Grad Survivor: Game Balance vs Real-World PhD Outcomes')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.legend()
ax.set_ylim(0, 75)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Add value labels
for bar in bars1:
    height = bar.get_height()
    if height > 1:
        ax.annotate(f'{height:.0f}%', xy=(bar.get_x() + bar.get_width()/2, height),
                    xytext=(0, 3), textcoords="offset points", ha='center', fontsize=9)
for bar in bars2:
    height = bar.get_height()
    ax.annotate(f'{height:.0f}%', xy=(bar.get_x() + bar.get_width()/2, height),
                xytext=(0, 3), textcoords="offset points", ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('balance_chart.png', dpi=150, bbox_inches='tight')
print("\nSaved: balance_chart.png")
plt.close()

Ending                   Game  Reality       Conf Source
-------------------------------------------------------------------------------------
defended                21.8%     63%       High   CGS PhD Completion Project
mastered_out            24.3%     17%     Medium   CGS + field estimates
left_for_industry       35.8%     10%    Low-Med   STEM career transition studies
abd                      5.1%      4%        Low   Attrition timing estimates
burned_out              13.0%      6%     Medium   Nature 2022 survey + PMC
broke                    0.0%      3%     Medium   Financial stress literature

Saved: balance_chart.png


## Results: By Archetype

In [ ]:
fig, axes = plt.subplots(1, len(ARCHETYPES_LIST), figsize=(16, 4), sharey=True)

for i, arch in enumerate(ARCHETYPES_LIST):
    ax = axes[i] if len(ARCHETYPES_LIST) > 1 else axes
    arch_results = [r for r in all_results if r['archetype'] == arch]
    arch_endings = Counter(r['ending'] for r in arch_results)
    arch_total = len(arch_results)
    
    pcts = [arch_endings.get(e, 0) / arch_total * 100 for e in endings_order]
    colors = ['#d4a017', '#e67e22', '#3498db', '#9b59b6', '#c0392b', '#95a5a6']
    
    ax.barh(range(len(endings_order)), pcts, color=colors)
    ax.set_yticks(range(len(endings_order)))
    if i == 0:
        ax.set_yticklabels(labels)
    else:
        ax.set_yticklabels([])
    ax.set_title(arch.replace('_', ' ').title(), fontsize=11)
    ax.set_xlim(0, 100)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_xlabel('Percentage')
    
    # Add percentage labels
    for j, pct in enumerate(pcts):
        if pct > 2:
            ax.text(pct + 1, j, f'{pct:.0f}%', va='center', fontsize=8)

fig.suptitle('Ending Distribution by Archetype (all strategies combined)', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('balance_by_archetype.png', dpi=150, bbox_inches='tight')
print("Saved: balance_by_archetype.png")
plt.close()

Saved: balance_by_archetype.png


## Results: By Strategy (Defend Rate Heatmap)

In [ ]:
# Defend rate heatmap
defend_matrix = np.zeros((len(ARCHETYPES_LIST), len(STRATEGIES)))

for i, arch in enumerate(ARCHETYPES_LIST):
    for j, strat in enumerate(STRATEGIES):
        subset = [r for r in all_results if r['archetype'] == arch and r['strategy'] == strat]
        defended = sum(1 for r in subset if r['ending'] == 'defended')
        defend_matrix[i, j] = defended / len(subset) * 100 if subset else 0

fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(defend_matrix, cmap='YlOrRd', aspect='auto', vmin=0, vmax=100)

ax.set_xticks(range(len(STRATEGIES)))
ax.set_xticklabels([s.replace('_', ' ').title() for s in STRATEGIES], rotation=30, ha='right')
ax.set_yticks(range(len(ARCHETYPES_LIST)))
ax.set_yticklabels([a.replace('_', ' ').title() for a in ARCHETYPES_LIST])

# Add text annotations
for i in range(len(ARCHETYPES_LIST)):
    for j in range(len(STRATEGIES)):
        val = defend_matrix[i, j]
        color = 'white' if val > 50 else 'black'
        ax.text(j, i, f'{val:.0f}%', ha='center', va='center', color=color, fontsize=11, fontweight='bold')

ax.set_title('Defend Rate: Archetype × Strategy', fontsize=13)
ax.set_xlabel('Strategy')
ax.set_ylabel('Archetype')
plt.colorbar(im, label='Defend %')
plt.tight_layout()
plt.savefig('defend_heatmap.png', dpi=150, bbox_inches='tight')
print("Saved: defend_heatmap.png")
plt.close()

Saved: defend_heatmap.png


## Detailed Breakdown by Archetype & Strategy

In [ ]:
for arch in ARCHETYPES_LIST:
    stats = ARCHETYPE_DATA[arch]
    print(f"\n{'='*85}")
    print(f"{arch.upper():24s} M:{stats['mind']:2d} B:{stats['body']:2d} W:{stats['wallet']:2d} Bo:{stats['bonds']:2d}")
    print(f"{'='*85}")
    print(f"{'Strategy':<13} {'Defend':>7} {'MstOut':>7} {'Indust':>7} {'ABD':>7} {'Burn':>7} {'Broke':>7} {'AvgSem':>7}")
    print('-' * 85)
    
    for strat in STRATEGIES:
        subset = [r for r in all_results if r['archetype'] == arch and r['strategy'] == strat]
        endings = Counter(r['ending'] for r in subset)
        n = len(subset)
        avg_sem = np.mean([r['semester'] for r in subset]) if subset else 0
        
        print(f"{strat:<13} "
              f"{endings.get('defended',0)/n*100:6.0f}% "
              f"{endings.get('mastered_out',0)/n*100:6.0f}% "
              f"{endings.get('left_for_industry',0)/n*100:6.0f}% "
              f"{endings.get('abd',0)/n*100:6.0f}% "
              f"{endings.get('burned_out',0)/n*100:6.0f}% "
              f"{endings.get('broke',0)/n*100:6.0f}% "
              f"{avg_sem:6.1f}")


OVERACHIEVER             M:50 B:45 W:65 Bo:55
Strategy       Defend  MstOut  Indust     ABD    Burn   Broke  AvgSem
-------------------------------------------------------------------------------------
random             2%     46%     10%     14%     28%      0%    5.3
left_bias          1%     55%     11%      7%     26%      0%    5.0
right_bias         4%     44%      4%     11%     38%      0%    5.2
safe               0%      1%     98%      0%      0%      0%    8.0
yolo               0%     59%      0%      6%     34%      0%    4.4

IMPOSTER                 M:35 B:55 W:65 Bo:60
Strategy       Defend  MstOut  Indust     ABD    Burn   Broke  AvgSem
-------------------------------------------------------------------------------------
random             1%     76%      2%      9%     12%      0%    4.5
left_bias          0%     74%      6%      8%     12%      0%    4.3
right_bias         2%     70%      3%     12%     12%      0%    4.6
safe               0%      4%     94%     

## Financial Stress Analysis

The v2 game implements a "financial stress" system where:
- Wallet has a **floor of 5** (you go into debt, not destitution)
- When Wallet < 20, each card drains **Mind -3, Body -2** (stress accumulation)
- This means money problems cause *burnout*, not direct *financial death*

This matches [real-world data](https://pmc.ncbi.nlm.nih.gov/articles/PMC6355122/): 62% of grad students report financial stress, but it's almost never the sole cause of departure. It amplifies burnout and mental health issues.

In [ ]:
# Analyze what actually kills players
causes = Counter()
for r in all_results:
    if r['ending'] not in ('defended', 'left_for_industry', 'stuck'):
        cause = r.get('cause')
        if cause is not None:  # Filter out None causes
            causes[cause] += 1

print("Direct causes of non-completion (excluding defended, industry, stuck):")
total_deaths = sum(causes.values())
if total_deaths > 0:
    for cause, count in causes.most_common():
        print(f"  {cause:<20} {count:>5} ({count/total_deaths*100:.1f}%)")
else:
    print("  (No deaths recorded)")

# Final wallet distribution for non-broke deaths
non_broke_wallets = [r['final_stats']['wallet'] for r in all_results 
                     if r['ending'] in ('burned_out', 'mastered_out', 'abd')]
if non_broke_wallets:
    print(f"\nWallet at death (burnout/mastered/ABD):")
    print(f"  Mean: {np.mean(non_broke_wallets):.1f}")
    print(f"  Median: {np.median(non_broke_wallets):.1f}")
    low_wallet_pct = sum(1 for w in non_broke_wallets if w < 20) / len(non_broke_wallets) * 100
    print(f"  % with wallet < 20 at death: {low_wallet_pct:.1f}%")
    print(f"  → This shows financial stress CAUSING burnout/exit even when wallet isn't at 0")
else:
    print("\n(No burnout/mastered/ABD endings to analyze)")

Direct causes of non-completion (excluding defended, industry, stuck):
  Mind                  1424 (67.1%)
  Body                   652 (30.7%)
  Bonds                   46 (2.2%)

Wallet at death (burnout/mastered/ABD):
  Mean: 67.6
  Median: 67.0
  % with wallet < 20 at death: 1.5%
  → This shows financial stress CAUSING burnout/exit even when wallet isn't at 0


## Left vs Right Choice Analysis

In [ ]:
all_cards = PHASE1_CARDS + PHASE2_CARDS + PHASE3_CARDS + UNIVERSAL_CARDS + ORGANIZER_CARDS

left_nets = []
right_nets = []
left_better = 0
right_better = 0
ties = 0

for card in all_cards:
    l_vals = list((card.get('eL') or {}).values()) if card.get('eL') else []
    r_vals = list((card.get('eR') or {}).values()) if card.get('eR') else []
    l_sum = sum(l_vals) if l_vals else 0
    r_sum = sum(r_vals) if r_vals else 0
    left_nets.append(l_sum)
    right_nets.append(r_sum)
    if l_sum > r_sum:
        left_better += 1
    elif r_sum > l_sum:
        right_better += 1
    else:
        ties += 1

if len(all_cards) > 0:
    print(f"Total non-milestone cards: {len(all_cards)}")
    print(f"Left is better:  {left_better}/{len(all_cards)} ({left_better/len(all_cards)*100:.0f}%)")
    print(f"Right is better: {right_better}/{len(all_cards)} ({right_better/len(all_cards)*100:.0f}%)")
    print(f"Tied:            {ties}/{len(all_cards)} ({ties/len(all_cards)*100:.0f}%)")
    print(f"\nAvg net effect:  Left={np.mean(left_nets):.1f}  Right={np.mean(right_nets):.1f}")
    print(f"\nInterpretation:")
    print(f"  Left choices tend toward engagement/action/social connection.")
    print(f"  Right choices tend toward caution/isolation/grinding alone.")
    print(f"  This is NOT a political left/right — it's collective action vs individualism.")
else:
    print("No cards loaded to analyze.")

Total non-milestone cards: 116
Left is better:  68/116 (59%)
Right is better: 41/116 (35%)
Tied:            7/116 (6%)

Avg net effect:  Left=1.8  Right=-5.5

Interpretation:
  Left choices tend toward engagement/action/social connection.
  Right choices tend toward caution/isolation/grinding alone.
  This is NOT a political left/right — it's collective action vs individualism.


## Summary & Limitations

### What this simulation shows:
1. The game's ending distribution roughly matches the *shape* of real PhD attrition (burnout and mastered-out dominate, financial death is rare)
2. Strategic play is heavily rewarded (~70-98% safe/yolo strategy vs ~10% random)
3. Each archetype creates a distinct play experience with different failure modes
4. The financial stress mechanic correctly routes money problems through burnout, not instant death

### Limitations:
1. **The "reality" numbers are estimates, not ground truth.** No single dataset breaks down PhD non-completion into the exact categories this game uses. The CGS completion rate (~63%) is solid; the breakdown of the ~37% who leave is synthesized from multiple overlapping sources.
2. **The game is intentionally harder than reality** for gameplay reasons. A 63% random win rate would make the game boring.
3. **Strategy definitions are simplistic.** Real players develop nuanced heuristics that don't map to "always pick the higher net sum."
4. **The simulation doesn't capture the narrative experience.** The game's impact comes from the *writing*, not the stat math. A card about your advisor ghosting you hits differently than "bonds:-15, mind:-5."

### Data sources (full citations):
- **CGS PhD Completion Project:** https://cgsnet.org/data-insights/diversity-equity-inclusiveness/degree-completion — completion rates by field, 49,000+ students across 30 universities
- **Nature 2022 Global Graduate Survey:** https://www.nature.com/articles/d41586-022-03394-0 — stress, satisfaction, financial strain, diversity data
- **PMC: Factors Affecting PhD Success:** https://pmc.ncbi.nlm.nih.gov/articles/PMC6355122/ — financial stress as amplifier, not sole cause
- **PMC: Leaving Academia:** https://pmc.ncbi.nlm.nih.gov/articles/PMC9534392/ — unhealthy research environments, attrition patterns
- **PNAS: PhD Careers in STEM:** https://www.pnas.org/doi/10.1073/pnas.1820079116 — industry transition rates, career outcomes
- **PMC: Academic Burnout:** https://pmc.ncbi.nlm.nih.gov/articles/PMC10163855/ — burnout prevalence in grad students
- **STEM Career Transitions:** https://www.tandfonline.com/doi/full/10.1080/21568235.2024.2404683 — post-PhD career data
- **MIT Living Wage Calculator:** https://livingwage.mit.edu/counties/13059 — Athens, GA cost of living
- **UGA Graduate School:** https://grad.uga.edu — stipend data, enrollment statistics